In [30]:
import pandas as pd
from backend import fx_fetcher, misc_fetcher, ticker_fetcher, base, api
from utils import utils
from bs4 import BeautifulSoup
import requests

import warnings
warnings.filterwarnings('ignore')

In [31]:
import yfinance as yf

In [32]:
db_conn = base.BaseDBConnector('core.db')
misc_fetcher_ = misc_fetcher.MiscFetcher(db_conn) 
misc_fetcher_.fetch_fst_trans_on_ticker('AAPL', 1)

TICKER            AAPL
DATE        2021-06-08
N_SHARES      3.830964
Name: 0, dtype: object

## PORTFOLIO STATS TESTING

In [35]:
DB_PATH = 'core.db'
REF_DATE = '2024-11-17'

In [36]:
stats = api.PortfolioStats(DB_PATH, REF_DATE)
df = stats.get_security_info('US', 'COUNTRY')
df

[{'COUNTRY': '',
  'TOTAL_VALUE': 130954.28857113031,
  'PROFIT': 44482.137978118684,
  'PROFIT%': 33.967683275952716}]

In [37]:
df_portfolio = stats.df_portfolio
df_portfolio = df_portfolio[df_portfolio.N_SHARES > 0]
df_portfolio

,TICKER,SECTOR,COUNTRY,FX,TOTAL_FEE,N_SHARES,TOTAL_COST,DT,PRICE,VALUE,TOTAL_VALUE,PROFIT,PROFIT%
0,AAPL,TECH,US,USD/RON,0.000000,7.958319,4642.896043,2024-11-17,225.000000,4.72290,8456.926991,3814.030948,82.147671
1,AMZN,RETAIL,US,USD/RON,0.000000,1.575731,1015.999963,2024-11-17,202.610001,4.72290,1507.827828,491.827865,48.408256
2,BRD,FINANCE,RO,RON,35.140000,387.000000,5292.180000,2024-11-17,18.900000,1.00000,7314.300000,2022.120000,38.209585
3,BRK-B,FINANCE,US,USD/RON,0.000000,5.778610,8025.808319,2024-11-17,470.279999,4.72290,12834.786771,4808.978452,59.918930
4,DIGI,TELECOMMUNICATION,RO,RON,11.110000,51.000000,1708.500000,2024-11-17,64.600000,1.00000,3294.600000,1586.100000,92.835821
5,EL,ENERGY,RO,RON,15.370000,224.000000,2315.860000,2024-11-17,13.260000,1.00000,2970.240000,654.380000,28.256458
7,H2O,ENERGY,RO,RON,88.750000,130.000000,16214.700000,2024-11-17,123.000000,1.00000,15990.000000,-224.700000,-1.385780
8,ISAC.L,ETF,GLOBAL,USD/RON,112.580380,185.530000,59176.436992,2024-11-17,88.889999,4.72290,77888.939376,18712.502384,31.621543
9,JPM,FINANCE,US,USD/RON,0.000000,0.623800,394.076917,2024-11-17,245.309998,4.72290,722.718814,328.641898,83.395369
10,M,HEALTH,RO,RON,14.740000,500.000000,2267.500000,2024-11-17,6.090000,1.00000,3045.000000,777.500000,34.288864


In [38]:
df_portfolio.TOTAL_VALUE.sum()

305848.9396385896

In [4]:
df_group = df.groupby('COUNTRY')
df_group = df_group.agg({
    'TOTAL_VALUE' : ['sum'],
    'PROFIT' : ['sum'],
}).reset_index()
df_group.columns = list(map(lambda x: x[0], df_group.columns))
df_group['PROFIT%'] = df_group['PROFIT'] / df_group['TOTAL_VALUE']
df_group['COUNTRY'] = ''
df_group

NameError: name 'df' is not defined

## Base

In [3]:
db_conn = base.BaseDBConnector('core.db')
ticker_fetcher_ = ticker_fetcher.TickerFetcher(db_conn)
fx_fetcher_ = fx_fetcher.FxFetcher(db_conn)

In [4]:
misc_fetcher_ = misc_fetcher.MiscFetcher(db_conn=db_conn)

In [9]:
misc_fetcher_.fetch_last_trans_on_ticker('SPY5.L', 5).to_dict(orient='records')

[{'TICKER': 'SPY5.L', 'DATE': '2022-12-02 13:46:11', 'N_SHARES': 2.0},
 {'TICKER': 'SPY5.L', 'DATE': '2022-11-07 05:29:13', 'N_SHARES': 2.0},
 {'TICKER': 'SPY5.L', 'DATE': '2022-10-10 17:13:19', 'N_SHARES': 3.0},
 {'TICKER': 'SPY5.L', 'DATE': '2022-09-01 12:50:57', 'N_SHARES': 3.0},
 {'TICKER': 'SPY5.L', 'DATE': '2022-08-01 04:18:29', 'N_SHARES': 1.0}]

In [7]:
portfolio_stats_ = api.PortfolioStats(DB_PATH, REF_DATE)

## Metrics & Period

In [3]:
period = api.Period('2Y')
period.frame_size, period.period_size, period.delta

('2', datetime.timedelta(days=365), datetime.timedelta(days=730))

In [4]:
api.DivYield(DB_PATH, '12M').compute()

0.0195

## New Features

In [6]:
DB_PATH = 'core.db'
REF_DATE = '2022-02-23'

In [10]:
class SecurityRichInfo:
    def __init__(self, db_path, ticker, ref_dt) -> None:
        self.db_conn = base.BaseDBConnector(db_path)
        self.fx_fetcher = fx_fetcher.FxFetcher(self.db_conn)
        self.misc_fetcher = misc_fetcher.MiscFetcher(self.db_conn)
        self.ticker_fetcher = ticker_fetcher.TickerFetcher(self.db_conn)

        self.ref_stats = api.PortfolioStats(db_path=db_path, ref_date=ref_dt, filter_kind='TICKER', filters=ticker)

In [11]:
security_info = SecurityRichInfo(db_path=DB_PATH, ticker='AAPL', ref_dt=REF_DATE)

In [21]:
security_info.ref_stats.df_portfolio.T[0].to_dict()

{'TICKER': 'AAPL',
 'SECTOR': 'TECH',
 'COUNTRY': 'US',
 'FX': 'USD/RON',
 'TOTAL_FEE': 0.0,
 'N_SHARES': 7.83096366,
 'TOTAL_COST': 4516.666664090667,
 'DT': '2022-02-23',
 'PRICE': 159.34912109375,
 'VALUE': 4.361199855804443,
 'TOTAL_VALUE': 5442.154538382483,
 'PROFIT': 925.4878742918163,
 'PROFIT%': 20.490506453571623}